In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Chemins
DATA_RAW = Path('../data/raw')
INCIDENTS_DIR = DATA_RAW / 'incidents'
MOBILISATION_DIR = DATA_RAW / 'mobilisation'

print("✅ Setup OK")
print(f"Dossier data : {DATA_RAW.resolve()}")

✅ Setup OK
Dossier data : C:\Users\guiblais\Documents\pyrefighter-v2\data\raw


In [2]:
print("=== INCIDENTS ===")
for f in sorted(INCIDENTS_DIR.glob('*')):
    print(f"  {f.name} — {f.stat().st_size / 1e6:.1f} Mo")

print("\n=== MOBILISATION ===")
for f in sorted(MOBILISATION_DIR.glob('*')):
    print(f"  {f.name} — {f.stat().st_size / 1e6:.1f} Mo")

=== INCIDENTS ===
  LFB Incident data from 2009 - 2017.csv — 328.6 Mo
  LFB Incident data from 2018 - 2023.xlsx — 153.4 Mo
  LFB Incident data from 2024 onwards.xlsx — 71.8 Mo

=== MOBILISATION ===
  LFB Mobilisation data from 2015 - 2020.xlsx — 118.0 Mo
  LFB Mobilisation data from 2021 - 2024.csv — 159.3 Mo
  LFB Mobilisation data from 2025.csv — 60.5 Mo
  LFB Mobilisation data from January 2009 - 2014.xlsx — 123.9 Mo


In [3]:
import time

file_2009_2017 = INCIDENTS_DIR / 'LFB Incident data from 2009 - 2017.csv'

print(f"📂 Rechargement avec le bon encoding (utf-8-sig)")
start = time.time()

df_inc_09_17 = pd.read_csv(
    file_2009_2017,
    low_memory=False,
    encoding='utf-8-sig'  # gère automatiquement le BOM
)

elapsed = time.time() - start
print(f"✅ Rechargé en {elapsed:.1f}s")
print(f"   Shape : {df_inc_09_17.shape[0]:,} lignes × {df_inc_09_17.shape[1]} colonnes")

# Vérif : les noms de colonnes doivent être propres maintenant
print(f"\n1ère colonne : '{df_inc_09_17.columns[0]}'")
print(f"Dernière colonne exemple : '{df_inc_09_17.columns[37]}'")

📂 Rechargement avec le bon encoding (utf-8-sig)
✅ Rechargé en 12.7s
   Shape : 988,279 lignes × 39 colonnes

1ère colonne : 'IncidentNumber'
Dernière colonne exemple : 'Notional Cost (£)'


In [4]:
print(f"=== RÉCAP ===")
print(f"Lignes         : {len(df_inc_09_17):,}")
print(f"Colonnes       : {df_inc_09_17.shape[1]}")
print(f"Période        : {df_inc_09_17['CalYear'].min()} → {df_inc_09_17['CalYear'].max()}")
print(f"Mémoire        : {df_inc_09_17.memory_usage(deep=True).sum() / 1e6:.1f} Mo")

print(f"\n=== Incidents par année ===")
print(df_inc_09_17['CalYear'].value_counts().sort_index())

print(f"\n=== Types d'incident (IncidentGroup) ===")
print(df_inc_09_17['IncidentGroup'].value_counts())

=== RÉCAP ===
Lignes         : 988,279
Colonnes       : 39
Période        : 2009 → 2017
Mémoire        : 524.0 Mo

=== Incidents par année ===
CalYear
2009    134379
2010    125060
2011    115132
2012    108391
2013    103055
2014     96079
2015     97831
2016    104838
2017    103514
Name: count, dtype: int64

=== Types d'incident (IncidentGroup) ===
IncidentGroup
False Alarm        481890
Special Service    299104
Fire               207285
Name: count, dtype: int64


In [5]:
import matplotlib.pyplot as plt

# Filtrer PumpOrder == 1 (premier camion arrivé = ce qui compte pour l'opérateur)
df_first_pump = df_mob_15_20[df_mob_15_20['PumpOrder'] == 1].copy()
print(f"📊 Après filtre PumpOrder==1 : {len(df_first_pump):,} lignes")
print(f"   ({len(df_first_pump)/len(df_mob_15_20)*100:.1f}% du dataset)")

print(f"\n=== Distribution AttendanceTimeSeconds ===")
print(df_first_pump['AttendanceTimeSeconds'].describe())

# Conversion en minutes pour lisibilité
df_first_pump['AttendanceTimeMinutes'] = df_first_pump['AttendanceTimeSeconds'] / 60

print(f"\n=== En minutes ===")
print(df_first_pump['AttendanceTimeMinutes'].describe().round(2))

# Histogramme
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution complète
axes[0].hist(df_first_pump['AttendanceTimeMinutes'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Attendance Time (minutes)')
axes[0].set_ylabel('Nombre d\'interventions')
axes[0].set_title('Distribution complète')
axes[0].axvline(df_first_pump['AttendanceTimeMinutes'].median(), color='red', linestyle='--', label=f"Médiane = {df_first_pump['AttendanceTimeMinutes'].median():.1f} min")
axes[0].legend()

# Zoom sur 0-20 min (là où sont 99% des interventions)
axes[1].hist(df_first_pump[df_first_pump['AttendanceTimeMinutes'] <= 20]['AttendanceTimeMinutes'], 
             bins=80, edgecolor='black', alpha=0.7, color='orange')
axes[1].set_xlabel('Attendance Time (minutes)')
axes[1].set_ylabel('Nombre d\'interventions')
axes[1].set_title('Zoom 0-20 minutes')
axes[1].axvline(6, color='red', linestyle='--', label='Objectif LFB : 6 min')
axes[1].legend()

plt.tight_layout()
plt.show()

NameError: name 'df_mob_15_20' is not defined

In [ ]:
print("=== Cas anormaux ===")
print(f"AttendanceTime = 0 : {(df_first_pump['AttendanceTimeSeconds'] == 0).sum():,}")
print(f"AttendanceTime = 1200 (20 min pile) : {(df_first_pump['AttendanceTimeSeconds'] == 1200).sum():,}")
print(f"AttendanceTime > 900 (15 min+) : {(df_first_pump['AttendanceTimeSeconds'] > 900).sum():,}")

# Vue rapide sur les 0
print("\n=== Exemples avec AttendanceTime = 0 ===")
print(df_first_pump[df_first_pump['AttendanceTimeSeconds'] == 0][
    ['IncidentNumber', 'CalYear', 'DateAndTimeMobilised', 'DateAndTimeArrived', 'DeployedFromStation_Name']
].head(10))

In [ ]:
# Filtre : PumpOrder == 1 + AttendanceTime propre (30s à 20 min exclus)
df_mob_clean = df_mob_15_20[
    (df_mob_15_20['PumpOrder'] == 1) &
    (df_mob_15_20['AttendanceTimeSeconds'] >= 30) &
    (df_mob_15_20['AttendanceTimeSeconds'] < 1200)
].copy()

print(f"Avant nettoyage : {len(df_mob_15_20):,}")
print(f"Après filtres (PumpOrder=1, 30s <= target < 20min) : {len(df_mob_clean):,}")
print(f"Réduction : -{(1 - len(df_mob_clean)/len(df_mob_15_20))*100:.1f}%")

# Ajout AttendanceTimeMinutes pour lisibilité
df_mob_clean['AttendanceTimeMinutes'] = df_mob_clean['AttendanceTimeSeconds'] / 60

# On ne garde que les colonnes utiles pour le merge (target + colonnes d'agrégats historiques)
mob_cols_keep = [
    'IncidentNumber',
    'AttendanceTimeSeconds',
    'AttendanceTimeMinutes',
    'DateAndTimeMobilised',
    'DeployedFromStation_Name',
    'PlusCode_Description',
    'TurnoutTimeSeconds',
    'TravelTimeSeconds',
    'PumpOrder'
]
df_mob_ready = df_mob_clean[mob_cols_keep]
print(f"\n✅ Mobilisation prête pour merge : {df_mob_ready.shape}")
df_mob_ready.head(3)

In [ ]:
# On filtre Incidents sur 2015-2017 (période overlap avec Mobilisation)
df_inc_15_17 = df_inc_09_17[df_inc_09_17['CalYear'].between(2015, 2017)].copy()
print(f"Incidents 2015-2017 : {len(df_inc_15_17):,}")
print(f"Mobilisation cleaned (2015-2020, filtrée sur 15-17 au merge) : {len(df_mob_ready):,}")

# MERGE
df = df_inc_15_17.merge(
    df_mob_ready,
    on='IncidentNumber',
    how='inner'  # on garde seulement les incidents qui ont un PumpOrder=1 associé
)

print(f"\n✅ Après merge : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")
print(f"   Taux de match : {len(df)/len(df_inc_15_17)*100:.1f}% des incidents 15-17")

In [ ]:
print("=== Répartition par année après merge ===")
print(df['CalYear'].value_counts().sort_index())

print("\n=== Distribution AttendanceTimeMinutes après merge ===")
print(df['AttendanceTimeMinutes'].describe().round(2))

print("\n=== Types d'incident dans le dataset mergé ===")
print(df['IncidentGroup'].value_counts())

In [ ]:
# Diagnostic 2015
print("=== Diagnostic 2015 ===\n")

# Combien d'incidents 2015 dans Incidents.csv ?
inc_2015 = df_inc_09_17[df_inc_09_17['CalYear'] == 2015]
print(f"Incidents.csv → 2015 : {len(inc_2015):,}")

# Combien de mobilisations 2015 (toutes PumpOrder) ?
mob_2015_all = df_mob_15_20[df_mob_15_20['CalYear'] == 2015]
print(f"Mobilisation.xlsx → 2015 (tous PumpOrder) : {len(mob_2015_all):,}")

# Combien de mobilisations 2015 avec PumpOrder == 1 ?
mob_2015_p1 = df_mob_15_20[(df_mob_15_20['CalYear'] == 2015) & (df_mob_15_20['PumpOrder'] == 1)]
print(f"Mobilisation.xlsx → 2015 avec PumpOrder=1 : {len(mob_2015_p1):,}")

# Dates min/max côté Mobilisation 2015
print(f"\nDates Mobilisation 2015 :")
print(f"  Min : {mob_2015_all['DateAndTimeMobilised'].min()}")
print(f"  Max : {mob_2015_all['DateAndTimeMobilised'].max()}")

# Répartition par mois côté Mobilisation 2015
print(f"\nMobilisation 2015 par mois :")
print(mob_2015_all.groupby(mob_2015_all['DateAndTimeMobilised'].dt.month).size())

# Répartition par mois côté Incidents 2015
inc_2015_dated = inc_2015.copy()
inc_2015_dated['DateOfCall_dt'] = pd.to_datetime(inc_2015_dated['DateOfCall'], format='%d-%b-%y', errors='coerce')
print(f"\nIncidents 2015 par mois :")
print(inc_2015_dated.groupby(inc_2015_dated['DateOfCall_dt'].dt.month).size())

In [ ]:
print("=== Formats IncidentNumber en 2015 ===\n")

# Échantillons de chaque source
print("📁 Incidents.csv (2015) — 5 exemples :")
print(df_inc_09_17[df_inc_09_17['CalYear'] == 2015]['IncidentNumber'].head(5).tolist())

print("\n📁 Mobilisation.xlsx (2015) — 5 exemples :")
print(df_mob_15_20[df_mob_15_20['CalYear'] == 2015]['IncidentNumber'].head(5).tolist())

# Longueurs typiques
print("\n=== Longueurs des IncidentNumber (2015) ===")
inc_lengths = df_inc_09_17[df_inc_09_17['CalYear'] == 2015]['IncidentNumber'].astype(str).str.len().value_counts()
mob_lengths = df_mob_15_20[df_mob_15_20['CalYear'] == 2015]['IncidentNumber'].astype(str).str.len().value_counts()
print(f"Incidents.csv : {inc_lengths.to_dict()}")
print(f"Mobilisation.xlsx : {mob_lengths.to_dict()}")

# Test : y a-t-il des IncidentNumber en commun ?
inc_ids_2015 = set(df_inc_09_17[df_inc_09_17['CalYear'] == 2015]['IncidentNumber'].astype(str))
mob_ids_2015 = set(df_mob_15_20[df_mob_15_20['CalYear'] == 2015]['IncidentNumber'].astype(str))
common = inc_ids_2015 & mob_ids_2015
print(f"\n=== Intersection ===")
print(f"IDs uniques Incidents 2015 : {len(inc_ids_2015):,}")
print(f"IDs uniques Mobilisation 2015 : {len(mob_ids_2015):,}")
print(f"IDs communs : {len(common):,}")
print(f"Match rate : {len(common) / len(inc_ids_2015) * 100:.1f}%")

# Comparons aussi 2016-2017 pour voir si c'est spécifique à 2015
for year in [2016, 2017]:
    inc_ids = set(df_inc_09_17[df_inc_09_17['CalYear'] == year]['IncidentNumber'].astype(str))
    mob_ids = set(df_mob_15_20[df_mob_15_20['CalYear'] == year]['IncidentNumber'].astype(str))
    common = inc_ids & mob_ids
    print(f"\n{year} : Incidents={len(inc_ids):,} | Mobilisation={len(mob_ids):,} | Match={len(common):,} ({len(common)/len(inc_ids)*100:.1f}%)")

In [ ]:
def clean_incident_number(x):
    """Nettoie l'IncidentNumber : gère les floats (ex: '1151.00' → '1151')"""
    if pd.isna(x):
        return None
    s = str(x).strip()
    # Si c'est un float genre "1151.00", on retire les ".00"
    if s.endswith('.00'):
        s = s[:-3]
    # Si c'est un float genre "1151.0", on retire les ".0"
    elif s.endswith('.0'):
        s = s[:-2]
    return s

# On applique le nettoyage sur les 2 sources
df_inc_09_17['IncidentNumber_clean'] = df_inc_09_17['IncidentNumber'].apply(clean_incident_number)
df_mob_15_20['IncidentNumber_clean'] = df_mob_15_20['IncidentNumber'].astype(str).str.strip()

# Vérif après nettoyage
print("=== Après nettoyage — 2015 ===")
inc_ids_2015 = set(df_inc_09_17[df_inc_09_17['CalYear'] == 2015]['IncidentNumber_clean'])
mob_ids_2015 = set(df_mob_15_20[df_mob_15_20['CalYear'] == 2015]['IncidentNumber_clean'])
common = inc_ids_2015 & mob_ids_2015
print(f"Match rate 2015 : {len(common) / len(inc_ids_2015) * 100:.1f}%")

# Sample après clean
print("\n=== Samples après clean ===")
print("Incidents 2015 :", df_inc_09_17[df_inc_09_17['CalYear'] == 2015]['IncidentNumber_clean'].head(5).tolist())
print("Mobilisation 2015 :", df_mob_15_20[df_mob_15_20['CalYear'] == 2015]['IncidentNumber_clean'].head(5).tolist())

# Idem 2016-2017 pour vérifier qu'on n'a pas cassé
for year in [2016, 2017]:
    inc_ids = set(df_inc_09_17[df_inc_09_17['CalYear'] == year]['IncidentNumber_clean'])
    mob_ids = set(df_mob_15_20[df_mob_15_20['CalYear'] == year]['IncidentNumber_clean'])
    common = inc_ids & mob_ids
    print(f"Match rate {year} : {len(common) / len(inc_ids) * 100:.1f}%")

In [ ]:
# Version cleaned de la Mobilisation prête pour merge
df_mob_ready = df_mob_clean[[
    'IncidentNumber',
    'AttendanceTimeSeconds',
    'AttendanceTimeMinutes',
    'DateAndTimeMobilised',
    'DeployedFromStation_Name',
    'PlusCode_Description',
    'TurnoutTimeSeconds',
    'TravelTimeSeconds',
    'PumpOrder'
]].copy()

# On ajoute la clé nettoyée
df_mob_ready['IncidentNumber_clean'] = df_mob_ready['IncidentNumber'].astype(str).str.strip()

# Merge sur la clé nettoyée
df_inc_15_17 = df_inc_09_17[df_inc_09_17['CalYear'].between(2015, 2017)].copy()

df = df_inc_15_17.merge(
    df_mob_ready.drop(columns=['IncidentNumber']),  # on retire la version brute
    on='IncidentNumber_clean',
    how='inner'
)

print(f"✅ Après merge sur clé cleaned : {df.shape[0]:,} lignes × {df.shape[1]} colonnes")

# Répartition par année
print("\n=== Répartition par année ===")
print(df['CalYear'].value_counts().sort_index())

In [ ]:
import time

INCIDENTS_DIR = project_root / 'data' / 'raw' / 'incidents'

# === Incidents 2018-2023 ===
print("📂 Chargement Incidents 2018-2023.xlsx (~3 min)...")
start = time.time()
df_inc_18_23 = pd.read_excel(
    INCIDENTS_DIR / 'LFB Incident data from 2018 - 2023.xlsx',
    engine='openpyxl'
)
print(f"✅ Chargé en {time.time()-start:.0f}s — {df_inc_18_23.shape}")

# === Incidents 2024+ ===
print("\n📂 Chargement Incidents 2024 onwards.xlsx (~1,5 min)...")
start = time.time()
df_inc_24_ = pd.read_excel(
    INCIDENTS_DIR / 'LFB Incident data from 2024 onwards.xlsx',
    engine='openpyxl'
)
print(f"✅ Chargé en {time.time()-start:.0f}s — {df_inc_24_.shape}")

# Vérif rapide périodes
print(f"\n=== Périodes ===")
print(f"2018-2023 : CalYear {sorted(df_inc_18_23['CalYear'].unique())}")
print(f"2024+     : CalYear {sorted(df_inc_24_['CalYear'].unique())}")

In [ ]:
# Comparaison des colonnes des 3 fichiers Incidents
cols_09_17 = set(df_inc_09_17.columns)
cols_18_23 = set(df_inc_18_23.columns)
cols_24_   = set(df_inc_24_.columns)

print("=== ANALYSE DES COLONNES ===")
print(f"2009-2017 : {len(cols_09_17)} colonnes")
print(f"2018-2023 : {len(cols_18_23)} colonnes")
print(f"2024+     : {len(cols_24_)} colonnes")

# Colonnes communes aux 3
common = cols_09_17 & cols_18_23 & cols_24_
print(f"\n✅ Colonnes communes aux 3 fichiers : {len(common)}")

# Colonnes uniquement dans 2018-2023 (nouvelles depuis 2018)
new_18 = cols_18_23 - cols_09_17
print(f"\n➕ Colonnes AJOUTÉES en 2018-2023 (n={len(new_18)}) :")
for c in sorted(new_18): print(f"    + {c}")

# Colonnes uniquement dans 2024+
new_24 = cols_24_ - cols_18_23
print(f"\n➕ Colonnes AJOUTÉES en 2024+ (n={len(new_24)}) :")
for c in sorted(new_24): print(f"    + {c}")

# Colonnes DISPARUES depuis 2009-2017
dropped = cols_09_17 - cols_18_23
print(f"\n➖ Colonnes SUPPRIMÉES depuis 2018 (n={len(dropped)}) :")
for c in sorted(dropped): print(f"    - {c}")

In [ ]:
MOBILISATION_DIR = project_root / 'data' / 'raw' / 'mobilisation'

# === Mobilisation 2009-2014 (XLSX) ===
print("📂 Chargement Mobilisation 2009-2014.xlsx (~2-3 min)...")
start = time.time()
df_mob_09_14 = pd.read_excel(
    MOBILISATION_DIR / 'LFB Mobilisation data from January 2009 - 2014.xlsx',
    engine='openpyxl'
)
print(f"✅ Chargé en {time.time()-start:.0f}s — {df_mob_09_14.shape}")

# === Mobilisation 2021-2024 (CSV) ===
print("\n📂 Chargement Mobilisation 2021-2024.csv (~15s)...")
start = time.time()
df_mob_21_24 = pd.read_csv(
    MOBILISATION_DIR / 'LFB Mobilisation data from 2021 - 2024.csv',
    low_memory=False,
    encoding='utf-8-sig'
)
print(f"✅ Chargé en {time.time()-start:.0f}s — {df_mob_21_24.shape}")

# === Mobilisation 2025 (CSV) ===
print("\n📂 Chargement Mobilisation 2025.csv (~5s)...")
start = time.time()
df_mob_25 = pd.read_csv(
    MOBILISATION_DIR / 'LFB Mobilisation data from 2025.csv',
    low_memory=False,
    encoding='utf-8-sig'
)
print(f"✅ Chargé en {time.time()-start:.0f}s — {df_mob_25.shape}")

# Vérif rapide des périodes
print(f"\n=== Périodes Mobilisation ===")
print(f"2009-2014 : CalYear {sorted(df_mob_09_14['CalYear'].unique())}")
print(f"2021-2024 : CalYear {sorted(df_mob_21_24['CalYear'].unique())}")
print(f"2025      : CalYear {sorted(df_mob_25['CalYear'].unique())}")

In [ ]:
# Comparaison des 4 fichiers Mobilisation
cols_09_14 = set(df_mob_09_14.columns)
cols_15_20 = set(df_mob_15_20.columns) - {'IncidentNumber_clean'}  # on retire notre colonne custom
cols_21_24 = set(df_mob_21_24.columns)
cols_25    = set(df_mob_25.columns)

print("=== ANALYSE DES COLONNES MOBILISATION ===")
print(f"2009-2014 : {len(cols_09_14)} colonnes")
print(f"2015-2020 : {len(cols_15_20)} colonnes")
print(f"2021-2024 : {len(cols_21_24)} colonnes")
print(f"2025      : {len(cols_25)} colonnes")

# Colonnes communes aux 4
common_mob = cols_09_14 & cols_15_20 & cols_21_24 & cols_25
print(f"\n✅ Colonnes communes aux 4 fichiers : {len(common_mob)}")

# Drift entre chaque paire successive
diffs = [
    ("2009-2014 → 2015-2020", cols_09_14, cols_15_20),
    ("2015-2020 → 2021-2024", cols_15_20, cols_21_24),
    ("2021-2024 → 2025",      cols_21_24, cols_25),
]

for label, before, after in diffs:
    added = after - before
    removed = before - after
    print(f"\n=== {label} ===")
    if added:
        print(f"  ➕ Ajoutées ({len(added)}) :")
        for c in sorted(added): print(f"      + {c}")
    if removed:
        print(f"  ➖ Retirées ({len(removed)}) :")
        for c in sorted(removed): print(f"      - {c}")
    if not added and not removed:
        print("  ✅ Aucun drift")

In [ ]:
# Retirer notre colonne custom du fichier 2009-2017 pour aligner
if 'IncidentNumber_clean' in df_inc_09_17.columns:
    df_inc_09_17_align = df_inc_09_17.drop(columns=['IncidentNumber_clean'])
else:
    df_inc_09_17_align = df_inc_09_17

# Concaténer les 3 fichiers Incidents
df_inc_all = pd.concat([df_inc_09_17_align, df_inc_18_23, df_inc_24_], ignore_index=True)

print(f"✅ Incidents concaténés : {df_inc_all.shape}")
print(f"\n=== Répartition par année ===")
print(df_inc_all['CalYear'].value_counts().sort_index())
print(f"\n=== Total : {len(df_inc_all):,} incidents sur {df_inc_all['CalYear'].nunique()} années ===")

# Libérer la mémoire des DataFrames intermédiaires
del df_inc_18_23, df_inc_24_
import gc; gc.collect()

In [ ]:
# Aligner sur 22 colonnes communes (drop BoroughName/WardName des fichiers récents)
cols_to_drop = ['BoroughName', 'WardName']
df_mob_21_24_align = df_mob_21_24.drop(columns=[c for c in cols_to_drop if c in df_mob_21_24.columns])
df_mob_25_align    = df_mob_25.drop(columns=[c for c in cols_to_drop if c in df_mob_25.columns])

# Concaténer les 4 fichiers Mobilisation
df_mob_all = pd.concat([df_mob_09_14, df_mob_15_20, df_mob_21_24_align, df_mob_25_align], ignore_index=True)

print(f"✅ Mobilisation concaténée : {df_mob_all.shape}")
print(f"\n=== Répartition par année ===")
print(df_mob_all['CalYear'].value_counts().sort_index())
print(f"\n=== Total : {len(df_mob_all):,} mobilisations sur {df_mob_all['CalYear'].nunique()} années ===")

# Libérer la mémoire
del df_mob_09_14, df_mob_15_20, df_mob_21_24, df_mob_25, df_mob_21_24_align, df_mob_25_align
gc.collect()

In [ ]:
# 1. Filtrer PumpOrder == 1 et target propre
print(f"Avant filtres : {len(df_mob_all):,}")

df_mob_all = df_mob_all[
    (df_mob_all['PumpOrder'] == 1) &
    (df_mob_all['AttendanceTimeSeconds'] >= 30) &
    (df_mob_all['AttendanceTimeSeconds'] < 1200)
].copy()

print(f"Après filtres (PumpOrder=1, 30s <= target < 20min) : {len(df_mob_all):,}")

# 2. Ajout AttendanceTimeMinutes
df_mob_all['AttendanceTimeMinutes'] = df_mob_all['AttendanceTimeSeconds'] / 60

# 3. Nettoyage clé de merge (fix bug 2015)
def clean_incident_number(x):
    if pd.isna(x):
        return None
    s = str(x).strip()
    if s.endswith('.00'):
        s = s[:-3]
    elif s.endswith('.0'):
        s = s[:-2]
    return s

df_inc_all['IncidentNumber_clean'] = df_inc_all['IncidentNumber'].apply(clean_incident_number)
df_mob_all['IncidentNumber_clean'] = df_mob_all['IncidentNumber'].astype(str).str.strip()

print(f"\n✅ Clés nettoyées, prêt pour merge")

In [ ]:
# On ne garde que les colonnes utiles de la Mobilisation pour éviter de dupliquer
mob_cols_keep = [
    'IncidentNumber_clean',
    'AttendanceTimeSeconds',
    'AttendanceTimeMinutes',
    'DateAndTimeMobilised',
    'DeployedFromStation_Name',
    'DeployedFromStation_Code',
    'DeployedFromLocation',
    'PlusCode_Description',
    'TurnoutTimeSeconds',
    'TravelTimeSeconds',
    'DelayCode_Description'
]
df_mob_ready = df_mob_all[mob_cols_keep]

# MERGE GLOBAL
print("🔥 Merge global en cours...")
start = time.time()

df_all = df_inc_all.merge(
    df_mob_ready,
    on='IncidentNumber_clean',
    how='inner'
)

print(f"✅ Merge terminé en {time.time()-start:.0f}s")
print(f"   Shape : {df_all.shape}")
print(f"   Taux de match : {len(df_all)/len(df_inc_all)*100:.1f}% des incidents")

# Répartition par année
print(f"\n=== Répartition par année du dataset final ===")
print(df_all['CalYear'].value_counts().sort_index())

In [ ]:
PROCESSED_DIR = project_root / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = PROCESSED_DIR / 'lfb_merged_2009_2026.parquet'

print(f"💾 Sauvegarde en Parquet : {parquet_path.name}")
start = time.time()
df_all.to_parquet(parquet_path, engine='pyarrow', compression='snappy', index=False)
print(f"✅ Sauvegardé en {time.time()-start:.0f}s")
print(f"   Taille du fichier : {parquet_path.stat().st_size / 1e6:.1f} Mo")
print(f"   (vs {(328+153+72+124+118+159+60):.0f} Mo de fichiers bruts = compression ~5-10x)")

# Test rechargement rapide
print("\n🧪 Test de rechargement...")
start = time.time()
df_test = pd.read_parquet(parquet_path)
print(f"✅ Rechargé en {time.time()-start:.1f}s — {df_test.shape}")

In [ ]:
print("🔧 Nettoyage des types avant Parquet...")

# 1. Convertir DateOfCall en datetime unifié
print("   • DateOfCall...")
# On tente le parsing avec plusieurs formats possibles
def parse_date_of_call(x):
    if pd.isna(x):
        return pd.NaT
    if isinstance(x, pd.Timestamp):
        return x
    # Format CSV "01-Apr-09"
    try:
        return pd.to_datetime(x, format='%d-%b-%y', errors='raise')
    except (ValueError, TypeError):
        # Fallback : parsing générique
        return pd.to_datetime(x, errors='coerce')

df_all['DateOfCall'] = df_all['DateOfCall'].apply(parse_date_of_call)

# 2. Détecter et convertir toutes les autres colonnes "object" mixtes
print("   • Détection colonnes mixtes restantes...")
problem_cols = []
for col in df_all.select_dtypes(include=['object']).columns:
    sample = df_all[col].dropna().head(1000)
    types = sample.apply(type).unique()
    if len(types) > 1:
        problem_cols.append((col, [t.__name__ for t in types]))

if problem_cols:
    print(f"   ⚠️ Colonnes à types mixtes trouvées :")
    for col, types in problem_cols:
        print(f"       - {col} : {types}")
        # Force en string pour être safe
        df_all[col] = df_all[col].astype(str).replace('nan', None)
else:
    print(f"   ✅ Aucune autre colonne mixte")

# 3. Vérif finale
print(f"\n=== Types finaux (colonnes datetime) ===")
for col in ['DateOfCall', 'DateAndTimeMobilised']:
    if col in df_all.columns:
        print(f"   {col} : {df_all[col].dtype}")

print(f"\n=== Aperçu DateOfCall ===")
print(df_all['DateOfCall'].head(3))
print(df_all['DateOfCall'].tail(3))

In [ ]:
PROCESSED_DIR = project_root / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = PROCESSED_DIR / 'lfb_merged_2009_2026.parquet'

print(f"💾 Sauvegarde en Parquet : {parquet_path.name}")
start = time.time()
df_all.to_parquet(parquet_path, engine='pyarrow', compression='snappy', index=False)
print(f"✅ Sauvegardé en {time.time()-start:.0f}s")
print(f"   Taille : {parquet_path.stat().st_size / 1e6:.1f} Mo")

# Test rechargement
print("\n🧪 Test de rechargement...")
start = time.time()
df_test = pd.read_parquet(parquet_path)
print(f"✅ Rechargé en {time.time()-start:.1f}s — {df_test.shape}")

In [ ]:
print("🔧 Nettoyage renforcé des types...")

# 1. TimeOfCall → forcer en string uniforme "HH:MM:SS"
print("   • TimeOfCall...")
def normalize_time(x):
    if pd.isna(x):
        return None
    # Si c'est déjà un str, on garde
    if isinstance(x, str):
        return x
    # Si c'est un datetime.time, on formate
    try:
        return x.strftime('%H:%M:%S')
    except AttributeError:
        return str(x)

df_all['TimeOfCall'] = df_all['TimeOfCall'].apply(normalize_time)

# 2. DateAndTimeMobilised → forcer en datetime
print("   • DateAndTimeMobilised...")
df_all['DateAndTimeMobilised'] = pd.to_datetime(df_all['DateAndTimeMobilised'], errors='coerce')

# 3. Balayer TOUTES les colonnes 'object' et vérifier sur toutes les lignes (pas juste 1000)
print("   • Audit complet des colonnes object...")
problem_cols = []
for col in df_all.select_dtypes(include=['object']).columns:
    # Échantillonne 10k lignes réparties sur tout le dataset
    sample_idx = np.linspace(0, len(df_all)-1, 10000, dtype=int)
    sample = df_all[col].iloc[sample_idx].dropna()
    types = set(type(v).__name__ for v in sample)
    if len(types) > 1:
        problem_cols.append((col, list(types)))

if problem_cols:
    print(f"\n   ⚠️ Colonnes mixtes détectées :")
    for col, types in problem_cols:
        print(f"       - {col} : {types}")
        # Force en string, gère les NaN proprement
        df_all[col] = df_all[col].astype(str).replace({'nan': None, 'NaT': None, 'None': None})


In [ ]:
print(f"💾 Sauvegarde en Parquet : {parquet_path.name}")
start = time.time()
df_all.to_parquet(parquet_path, engine='pyarrow', compression='snappy', index=False)
print(f"✅ Sauvegardé en {time.time()-start:.0f}s")
print(f"   Taille : {parquet_path.stat().st_size / 1e6:.1f} Mo")

# Test rechargement
print("\n🧪 Test de rechargement...")
start = time.time()
df_test = pd.read_parquet(parquet_path)
print(f"✅ Rechargé en {time.time()-start:.1f}s — {df_test.shape}")
del df_test

In [ ]:
# Cleanup de la RAM
for var in ['df_all', 'df_test', 'df_inc_all', 'df_mob_all', 'df_inc_09_17', 
            'df_inc_09_17_align', 'df_mob_ready', 'df_mob_clean', 'df_mob_15_20',
            'df_first_pump', 'df_inc_15_17', 'df', 'df_sample', 'df_inc_18_23', 
            'df_inc_24_', 'df_mob_09_14', 'df_mob_21_24', 'df_mob_25']:
    if var in dir():
        try:
            del globals()[var]
        except KeyError:
            pass

import gc
gc.collect()
print("✅ RAM libérée")

# Rechargement propre du Parquet
import pandas as pd
import numpy as np
from pathlib import Path
import time

project_root = Path('..').resolve()
parquet_path = project_root / 'data' / 'processed' / 'lfb_merged_2009_2026.parquet'

print(f"\n📂 Rechargement depuis Parquet...")
start = time.time()
df = pd.read_parquet(parquet_path)
print(f"✅ Chargé en {time.time()-start:.1f}s")
print(f"   Shape : {df.shape}")
print(f"   Mémoire : {df.memory_usage(deep=True).sum() / 1e6:.1f} Mo")
print(f"   Période : {df['CalYear'].min()} → {df['CalYear'].max()}")

In [ ]:
%pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl.metadata (41 kB)
  Using cached idna-3.18-py3-none-any.whl.metadata (6.1 kB)
  Using cached urllib3-2.7.0-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.6.17-py3-none-any.whl.metadata (2.5 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached charset_normalizer-3.4.7-cp314-cp314-win_amd64.whl (159 kB)
Using cached idna-3.18-py3-none-any.whl (65 kB)
Using cached urllib3-2.7.0-py3-none-any.whl (131 kB)
Using cached certifi-2026.6.17-py3-none-any.whl (133 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ----------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
=== ENRICHISSEMENT #1 : MÉTÉO via Open-Meteo API ===
Historique quotidien complet 2009-2026 pour Londres.
"""

import requests
import pandas as pd
from pathlib import Path

WEATHER_DIR = project_root / 'data' / 'raw' / 'weather'
WEATHER_DIR.mkdir(parents=True, exist_ok=True)
weather_cache = WEATHER_DIR / 'london_daily_2009_2026.csv'

if weather_cache.exists():
    print(f"✅ Cache trouvé : {weather_cache.name}")
    df_weather = pd.read_csv(weather_cache, parse_dates=['date'])
else:
    print(f"🌦️  Téléchargement météo Londres 2009-2026 via Open-Meteo...")
    
    # Coordonnées : Central London (Charing Cross approximatif)
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": 51.5074,
        "longitude": -0.1278,
        "start_date": "2009-01-01",
        "end_date": "2026-06-30",
        "daily": ",".join([
            "temperature_2m_max", "temperature_2m_min", "temperature_2m_mean",
            "precipitation_sum", "rain_sum", "snowfall_sum",
            "wind_speed_10m_max", "wind_gusts_10m_max",
            "sunshine_duration"
        ]),
        "timezone": "Europe/London"
    }
    
    start = time.time()
    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    print(f"✅ Reçu en {time.time()-start:.1f}s")
    
    # Conversion en DataFrame
    df_weather = pd.DataFrame(data['daily'])
    df_weather['date'] = pd.to_datetime(df_weather['time'])
    df_weather = df_weather.drop(columns=['time'])
    
    # Sauvegarde cache
    df_weather.to_csv(weather_cache, index=False)
    print(f"💾 Sauvegardé : {weather_cache.name}")

print(f"\n=== Aperçu météo Londres ===")
print(f"Shape : {df_weather.shape}")
print(f"Période : {df_weather['date'].min().date()} → {df_weather['date'].max().date()}")
print(f"\n{df_weather.head()}")

NameError: name 'project_root' is not defined

In [ ]:
"""
Étape 2 : créer les features météo à joindre sur DateOfCall
"""

# Renommage propre + features dérivées
df_weather = df_weather.rename(columns={
    'temperature_2m_max': 'temp_max',
    'temperature_2m_min': 'temp_min',
    'temperature_2m_mean': 'temp_mean',
    'precipitation_sum': 'precip_mm',
    'rain_sum': 'rain_mm',
    'snowfall_sum': 'snow_cm',
    'wind_speed_10m_max': 'wind_max_kmh',
    'wind_gusts_10m_max': 'wind_gust_max_kmh',
    'sunshine_duration': 'sunshine_sec'
})

# Features binaires météo
df_weather['is_rainy']     = (df_weather['rain_mm'] > 5).astype(int)    # pluie significative
df_weather['is_snowy']     = (df_weather['snow_cm'] > 0).astype(int)    # neige au sol
df_weather['is_windy']     = (df_weather['wind_gust_max_kmh'] > 60).astype(int)  # tempête
df_weather['is_heatwave']  = (df_weather['temp_max'] > 28).astype(int)  # canicule (rare Londres)
df_weather['is_freezing']  = (df_weather['temp_min'] < 0).astype(int)   # gel

print(f"=== Statistiques météo Londres 2009-2026 ===")
print(f"Nombre de jours pluvieux (>5mm)      : {df_weather['is_rainy'].sum()} ({df_weather['is_rainy'].mean()*100:.1f}%)")
print(f"Nombre de jours neigeux              : {df_weather['is_snowy'].sum()} ({df_weather['is_snowy'].mean()*100:.1f}%)")
print(f"Nombre de jours de tempête (>60km/h) : {df_weather['is_windy'].sum()} ({df_weather['is_windy'].mean()*100:.1f}%)")
print(f"Nombre de jours de canicule (>28°C)  : {df_weather['is_heatwave'].sum()} ({df_weather['is_heatwave'].mean()*100:.1f}%)")
print(f"Nombre de jours de gel (<0°C)        : {df_weather['is_freezing'].sum()} ({df_weather['is_freezing'].mean()*100:.1f}%)")

# Prévisualisation
print(f"\n{df_weather[['date', 'temp_max', 'temp_min', 'precip_mm', 'snow_cm', 'is_snowy', 'is_heatwave']].head()}")

In [ ]:
"""
Étape 3 : joindre la météo sur le dataset LFB via DateOfCall
"""

# Extraire seulement la date (sans heure) côté df
df['date_join'] = df['DateOfCall'].dt.normalize()

# Merge avec la météo
weather_cols = ['date', 'temp_max', 'temp_min', 'temp_mean', 'precip_mm', 'rain_mm',
                'snow_cm', 'wind_max_kmh', 'wind_gust_max_kmh', 'sunshine_sec',
                'is_rainy', 'is_snowy', 'is_windy', 'is_heatwave', 'is_freezing']

df = df.merge(
    df_weather[weather_cols].rename(columns={'date': 'date_join'}),
    on='date_join',
    how='left'
)

print(f"✅ Merge météo effectué")
print(f"   Lignes avec météo NaN : {df['temp_mean'].isna().sum():,} / {len(df):,}")

# CORRÉLATIONS AVEC LA TARGET
print(f"\n=== CORRÉLATIONS MÉTÉO ↔ AttendanceTimeMinutes ===")
weather_features = ['temp_max', 'temp_min', 'temp_mean', 'precip_mm', 'rain_mm', 'snow_cm',
                    'wind_max_kmh', 'wind_gust_max_kmh',
                    'is_rainy', 'is_snowy', 'is_windy', 'is_heatwave', 'is_freezing']

for col in weather_features:
    corr = df[col].corr(df['AttendanceTimeMinutes'])
    print(f"  {col:<25} : {corr:+.4f}")

# Impact concret des événements extrêmes
print(f"\n=== Impact NEIGE sur target ===")
print(df.groupby('is_snowy')['AttendanceTimeMinutes'].agg(['mean', 'median', 'count']).round(2))

print(f"\n=== Impact TEMPÊTE sur target ===")
print(df.groupby('is_windy')['AttendanceTimeMinutes'].agg(['mean', 'median', 'count']).round(2))